In [ ]:
///- --package .

#### block_plan (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:| run_outcome =
    | Completed
    | Crashed

:| run_fate =
    | Counted
    | Rerun
    | Missing

:== progress = { stage : i32; run : i32; retried : bool; counted : i32; missing : i32 }

:= outcome_of_experience (experience : string) : run_outcome = ? experience = "" ?> Crashed |: Completed

:= fate_of (retried : bool) (o : run_outcome) : run_fate =
    ?= o ?|
    | Completed => Counted
    | Crashed => ? retried ?> Missing |: Rerun

:= starting () : progress = { stage = 0; run = 0; retried = !-; counted = 0; missing = 0 }

:= finished (runs : array i32) (p : progress) : bool = p::stage >= length runs

:= next_slot (runs : array i32) (p : progress) : i32 * i32 =
    ? p::run + 1 >= index runs p::stage ?> p::stage + 1, 0 |: p::stage, p::run + 1

:= advanced (runs : array i32) (p : progress) (counted : i32) (missing : i32) : progress =
    := stage, run = next_slot runs p
    { stage = stage; run = run; retried = !-; counted = counted; missing = missing }

:= after (runs : array i32) (p : progress) (o : run_outcome) : progress =
    ?= fate_of p::retried o ?|
    | Rerun => { p ?| retried = !+ }
    | Counted => advanced runs p (p::counted + 1) p::missing
    | Missing => advanced runs p p::counted (p::missing + 1)

:| stage_step =
    | InstallProbe : string
    | Runs : i32
    | RemoveProbe
    | KeepTrace : string

:= probed_stage (probe : string) (runs : i32) (trace : string) : list stage_step =
    [ InstallProbe probe; Runs runs; RemoveProbe; KeepTrace trace ]

:= plain_stage (runs : i32) : list stage_step = [ Runs runs ]

:= runs_of (steps : list stage_step) : i32 =
    listm::fold (\ total step => ?= step ?| Runs n => total + n | _ => total) 0 steps

:= probe_left_installed (steps : list stage_step) : bool =
    listm::fold
        \ installed step =>
            ?= step ?|
            | InstallProbe _ => !+
            | RemoveProbe => !-
            | _ => installed
        !-
        steps

:= stages_of (first : i32) (second : i32) : array i32 =
    := runs : array i32 = create 2
    set runs 0 first
    set runs 1 second
    runs

:= block_progress (first : i32) (outcomes : list run_outcome) : progress =
    listm::fold (\ p o => after (stages_of first 1) p o) (starting ()) outcomes

:= block_finished (first : i32) (outcomes : list run_outcome) : bool =
    finished (stages_of first 1) (block_progress first outcomes)

#### an_empty_experience_reading_means_the_run_crashed (test)

In [1]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~empty = ""
(outcome_of_experience empty, outcome_of_experience "675") |> _assert_eq (Crashed, Completed)

.c (c)
__assert_eq { expected = (Crashed, Completed) }
.lua (lua)
__assert_eq { expected = (Crashed, Completed) }
.gleam (gleam)
__assert_eq { expected = (Crashed, Completed) }

#### a_crash_is_rerun_once_then_counted_missing (test)

In [2]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~retried = !+
(fate_of (not retried) Completed, fate_of (not retried) Crashed, fate_of retried Crashed, fate_of retried Completed)
|> _assert_eq (Counted, Rerun, Missing, Counted)

.c (c)
__assert_eq { expected = (Counted, Rerun, Missing, Counted) }
.lua (lua)
__assert_eq { expected = (Counted, Rerun, Missing, Counted) }
.gleam (gleam)
__assert_eq { expected = (Counted, Rerun, Missing, Counted) }

#### the_first_completed_run_counts_and_moves_on (test)

In [3]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~first = 2i32
:= p = block_progress first [ Completed ]
(p::stage, p::run, p::counted) |> _assert_eq (0i32, 1i32, 1i32)

.c (c)
__assert_eq { expected = (0, 1, 1) }
.lua (lua)
__assert_eq { expected = (0, 1, 1) }
.gleam (gleam)
__assert_eq { expected = (0, 1, 1) }

#### a_crash_marks_the_run_for_a_rerun (test)

In [4]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~first = 2i32
:= p = block_progress first [ Completed; Crashed ]
(p::stage, p::run, p::retried) |> _assert_eq (0i32, 1i32, !+)

.c (c)
__assert_eq { expected = (0, 1, true) }
.lua (lua)
__assert_eq { expected = (0, 1, true) }
.gleam (gleam)
__assert_eq { expected = (0, 1, true) }

#### a_completed_rerun_moves_to_the_next_stage (test)

In [5]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~first = 2i32
:= p = block_progress first [ Completed; Crashed; Completed ]
(p::stage, p::run, p::counted, p::retried) |> _assert_eq (1i32, 0i32, 2i32, !-)

.c (c)
__assert_eq { expected = (1, 0, 2, false) }
.lua (lua)
__assert_eq { expected = (1, 0, 2, false) }
.gleam (gleam)
__assert_eq { expected = (1, 0, 2, false) }

#### a_block_runs_every_stage_in_order (test)

In [6]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~first = 2i32
:= p = block_progress first [ Completed; Crashed; Completed; Completed ]
(p::stage, p::counted, p::missing) |> _assert_eq (2i32, 3i32, 0i32)

.c (c)
__assert_eq { expected = (2, 3, 0) }
.lua (lua)
__assert_eq { expected = (2, 3, 0) }
.gleam (gleam)
__assert_eq { expected = (2, 3, 0) }

#### a_block_ends_after_its_last_stage (test)

In [7]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~first = 2i32
:= three, four = [ Completed; Crashed; Completed ], [ Completed; Crashed; Completed; Completed ]
(block_finished first three, block_finished first four) |> _assert_eq (!-, !+)

.c (c)
__assert_eq { expected = (false, true) }
.lua (lua)
__assert_eq { expected = (false, true) }
.gleam (gleam)
__assert_eq { expected = (false, true) }

#### two_crashes_count_one_run_missing (test)

In [8]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~first = 1i32
:= runs = stages_of first 1
:= p = after runs (after runs (starting ()) Crashed) Crashed
(p::stage, p::counted, p::missing) |> _assert_eq (1i32, 0i32, 1i32)

.c (c)
__assert_eq { expected = (1, 0, 1) }
.lua (lua)
__assert_eq { expected = (1, 0, 1) }
.gleam (gleam)
__assert_eq { expected = (1, 0, 1) }

#### a_probed_stage_keeps_its_run_count (test)

In [9]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~runs = 10i32
(runs_of (probed_stage "lead_trace.lua" runs "lead-trace.log"), runs_of (plain_stage runs))
|> _assert_eq (10i32, 10i32)

.c (c)
__assert_eq { expected = (10, 10) }
.lua (lua)
__assert_eq { expected = (10, 10) }
.gleam (gleam)
__assert_eq { expected = (10, 10) }

#### a_probed_stage_removes_its_probe_and_keeps_the_trace (test)

In [10]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~runs = 10i32
:= steps = probed_stage "lead_trace.lua" runs "lead-trace.log"
probe_left_installed steps |> _assert_eq !-

.c (c)
__assert_eq { expected = false }
.lua (lua)
__assert_eq { expected = false }
.gleam (gleam)
__assert_eq { expected = false }

#### a_stage_without_removal_leaves_its_probe_installed (test)

In [11]:
///- --test --symbol
///> c
///> lua
///> gleam

:= ~runs = 10i32
probe_left_installed [ InstallProbe "lead_trace.lua"; Runs runs ] |> _assert_eq !+

.c (c)
__assert_eq { expected = true }
.lua (lua)
__assert_eq { expected = true }
.gleam (gleam)
__assert_eq { expected = true }